This notebook is to extract the preferred foot of every player and concatenate it to the shots dataset as a feature.

The idea is that, for everyshot, we believe the foot the shot was taken with, as well as the preferred foot should be an even better predictor of shot success. That is, xG should include the likelihood of a goal being scored, given a shot was taken with the preferred foot(or not). i.e., P[Goal|Shot taken with preferred foot] = 1 - P[Goal | Shot not taken with preferred foot]

This notebook will hence do the following, get a dataset of all FIFA 16 players, from the FIFA index, that includes the players preferred foot. Do some data preprocessing that validates the data. Then implement a matching algorithm that matches each player from the new dataset with statsbomb dataset. 

## DATA PREPROCESSING

For the preferred foot feature. I will use the sofifa dataset(found here: https://www.kaggle.com/datasets/luisfucros/fifa-players?resource=download). These datasets contain detailed player statistics and attributes for football players as featured in the Sofifa database, which is a widely recognized source of football player data used for simulation and analysis purposes. The dataset includes player-specific information such as personal details, club affiliations, performance metrics, and skill ratings, providing a comprehensive overview of player profiles within the world of football.

In [1]:
import pandas as pd

df = pd.read_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/Notebooks/players_16.csv")
df.head()

/var/folders/lz/th4y3_7n34lfmhmw0b_t5y0r0000gn/T/ipykernel_88581/913275278.py:3: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/Notebooks/players_16.csv")


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",94,95,111000000.0,550000.0,28,...,44+3,44+3,44+3,57+3,19+3,https://cdn.sofifa.net/players/158/023/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",93,93,85500000.0,475000.0,30,...,52+3,52+3,52+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/16_120.png,https://cdn.sofifa.net/teams/243/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",90,90,56000000.0,250000.0,31,...,47+3,47+3,47+3,59+3,19+3,https://cdn.sofifa.net/players/009/014/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/105035/60.png,https://cdn.sofifa.net/flags/nl.png
3,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,GK,90,90,58000000.0,250000.0,29,...,33+3,33+3,33+3,33+3,87+3,https://cdn.sofifa.net/players/167/495/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1337/60.png,https://cdn.sofifa.net/flags/de.png
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,90,90,69000000.0,300000.0,28,...,58+3,58+3,58+3,64+3,37+3,https://cdn.sofifa.net/players/176/580/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,NaN,https://cdn.sofifa.net/flags/uy.png


In [2]:
df.rename(columns={'player_url': "Source"}, inplace=True)
df = df[["sofifa_id", "short_name", "long_name", "preferred_foot", "Source"]].copy()
df.head()

,sofifa_id,short_name,long_name,preferred_foot,Source
0,158023,L. Messi,Lionel Andrés Messi Cuccittini,Left,https://sofifa.com/player/158023/lionel-messi/...
1,20801,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,Right,https://sofifa.com/player/20801/c-ronaldo-dos-...
2,9014,A. Robben,Arjen Robben,Left,https://sofifa.com/player/9014/arjen-robben/16...
3,167495,M. Neuer,Manuel Peter Neuer,Right,https://sofifa.com/player/167495/manuel-neuer/...
4,176580,L. Suárez,Luis Alberto Suárez Díaz,Right,https://sofifa.com/player/176580/luis-suarez/1...


In [3]:
df = pd.get_dummies(df, columns=['preferred_foot'])
df[['preferred_foot_Left', 'preferred_foot_Right']] = df[['preferred_foot_Left', 'preferred_foot_Right']].astype(int)
df.head()

,sofifa_id,short_name,long_name,Source,preferred_foot_Left,preferred_foot_Right
0,158023,L. Messi,Lionel Andrés Messi Cuccittini,https://sofifa.com/player/158023/lionel-messi/...,1,0
1,20801,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,https://sofifa.com/player/20801/c-ronaldo-dos-...,0,1
2,9014,A. Robben,Arjen Robben,https://sofifa.com/player/9014/arjen-robben/16...,1,0
3,167495,M. Neuer,Manuel Peter Neuer,https://sofifa.com/player/167495/manuel-neuer/...,0,1
4,176580,L. Suárez,Luis Alberto Suárez Díaz,https://sofifa.com/player/176580/luis-suarez/1...,0,1


In [ ]:
shots = pd.read_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/shots_extended.csv")
shots.head()